# FineWeb cached document lengths

**Hypothesis:** Document lengths in the existing FineWeb cache have a long right tail. We expect the mean to exceed the median and the upper percentiles to be much larger than the median. A compact distribution or a sharp cutoff would instead suggest more uniform lengths or effects of cache-construction filtering.

**Scope and conclusions:** Scan every document in the selected cache, without additional filtering, sampling, shuffling, or truncation. Report document count, mean, population standard deviation (`ddof=0`), minimum, percentiles (1, 5, 25, 50, 75, 95, 99), and maximum in two units: cached GPT-2 tokens and Unicode characters (`len(text)`). Percentiles use NumPy's linear interpolation. These summaries describe this cache only; they do not establish lengths for all FineWeb, Qwen token counts, or exact training truncation rates. Construction filters may already have restricted the cache.

**Prerequisites:** Use the `stego` Conda environment with the repository dependencies, NumPy, Jupyter, and an already completed FineWeb cache. Export `STEGO_ARTIFACTS_DIR` pointing to your existing artifacts directory before starting Jupyter. The default cache is `$STEGO_ARTIFACTS_DIR/datasets/fineweb/fineweb-500k`.

**Run:** From the repository root, run `conda activate stego`, then `PYTHONPATH="$PWD" jupyter lab ciphers/kirchenbauer_et_al/experiment/E20260916_fineweb_document_lengths.ipynb`. Select a kernel using `stego`, edit `CACHE_NAME` if needed, and run all cells. The shared loader validates the completion marker, manifest, Parquet schemas, part sequence, and row counts. Missing or invalid caches raise an error; this notebook does not build or download a cache or tokenizer.

**Outputs:** Statistics are printed below; no separate output files are written. Only integer lengths are retained for exact percentiles, so memory usage scales with the number of documents rather than their combined text size.

In [1]:
import os
from pathlib import Path

import numpy as np

from ciphers.kirchenbauer_et_al.src.cache_fineweb import load_fineweb_cache

CACHE_NAME = "fineweb-500k"

/opt/miniconda3/envs/stego/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The loader returns document dictionaries. This notebook consumes two required keys: `token_count`, the stored integer GPT-2 token count for the document, and `text`, the extracted document string whose Unicode character count is measured. Other cached fields are unused. The resulting `document_lengths` array has shape `(documents, 2)`, with GPT-2 token counts in column 0 and character counts in column 1; each document contributes once with equal weight.

In [3]:
cached_documents = load_fineweb_cache(CACHE_NAME, shuffle=False)
cache_directory = Path(os.environ["STEGO_ARTIFACTS_DIR"]) / "datasets" / "fineweb" / CACHE_NAME
print(f"Cache: {cache_directory}")

# Keep lengths only so the full corpus text need not fit in memory.
document_lengths = np.asarray(
    [(document["token_count"], len(document["text"])) for document in cached_documents],
    dtype=np.int64,
)
print(f"Documents: {len(document_lengths):,}")

Cache: /Users/4gate/git/StegoICMLMechInterp2026/artifacts/datasets/fineweb/fineweb-500k
Documents: 500,000


In [4]:
percentiles = (0, 1, 5, 25, 50, 75, 95, 99, 100)
statistic_labels = ("Minimum", "p01", "p05", "p25", "Median (p50)", "p75", "p95", "p99", "Maximum")

for column_index, unit in enumerate(("GPT-2 tokens (cached)", "Unicode characters")):
    lengths = document_lengths[:, column_index]
    print(f"\n{unit}")
    print(f"  {'Mean':<22} {lengths.mean():>14,.2f}")
    print(f"  {'Std. dev. (population)':<22} {lengths.std(ddof=0):>14,.2f}")
    for label, value in zip(statistic_labels, np.percentile(lengths, percentiles, method="linear"), strict=True):
        print(f"  {label:<22} {value:>14,.2f}")


GPT-2 tokens (cached)
  Mean                           693.49
  Std. dev. (population)       1,384.47
  Minimum                         33.00
  p01                             74.00
  p05                             99.00
  p25                            205.00
  Median (p50)                   404.00
  p75                            773.00
  p95                          2,000.00
  p99                          4,891.02
  Maximum                    130,040.00

Unicode characters
  Mean                         3,059.60
  Std. dev. (population)       5,979.06
  Minimum                        136.00
  p01                            301.00
  p05                            414.00
  p25                            892.00
  Median (p50)                 1,786.00
  p75                          3,450.00
  p95                          8,839.00
  p99                         21,469.00
  Maximum                    522,573.00
